# 📊 Notebook 04 — Gold Aggregations & KPI Reporting

**Goal:** Build pre-aggregated Gold tables and KPI views optimized for Power BI Direct Lake.

> **Run time:** ~5 min

These Gold tables power the executive dashboard — branch performance, product profitability, customer segment analysis, and monthly trends.

In [ ]:
from pyspark.sql import functions as F

fact_txn   = spark.table('fact_transactions')
fact_loans = spark.table('fact_loans')
print('Loaded fact tables')

## Gold Table 1: Branch Performance KPIs

In [ ]:
gold_branch_perf = fact_loans.groupBy('BranchID','BranchName','Region') \
    .agg(
        F.count('LoanID').alias('TotalLoans'),
        F.round(F.sum('LoanAmount'), 0).alias('TotalLoanVolume'),
        F.round(F.avg('LoanAmount'), 0).alias('AvgLoanSize'),
        F.round(F.avg('InterestRate'), 2).alias('AvgInterestRate'),
        F.sum('IsDefault').alias('TotalDefaults'),
        F.round(F.sum('IsDefault') * 100.0 / F.count('LoanID'), 1).alias('DefaultRate_Pct'),
        F.round(F.sum('OutstandingBalance'), 0).alias('TotalOutstanding')
    )

gold_branch_perf.write.format('delta').mode('overwrite').saveAsTable('gold_branch_performance')
print(f'gold_branch_performance: {gold_branch_perf.count()} rows')
gold_branch_perf.orderBy('TotalLoanVolume', ascending=False).show()

## Gold Table 2: Monthly Transaction Trends

In [ ]:
gold_monthly = fact_txn.filter(F.col('Year').isNotNull()) \
    .groupBy('Year','MonthName','Month','Quarter') \
    .agg(
        F.count('TransactionID').alias('NumTransactions'),
        F.round(F.sum('Amount'), 0).alias('TotalAmount'),
        F.round(F.avg('Amount'), 2).alias('AvgTransactionAmount'),
        F.countDistinct('CustomerID').alias('UniqueCustomers'),
        F.countDistinct('AccountID').alias('ActiveAccounts'),
        F.sum(F.when(F.col('Status') == 'Failed', 1).otherwise(0)).alias('FailedTransactions'),
        F.sum(F.when(F.col('IsLargeTransaction') == True, F.col('Amount')).otherwise(0)).alias('LargeTransactionVolume')
    ) \
    .orderBy('Year','Month')

gold_monthly.write.format('delta').mode('overwrite').saveAsTable('gold_monthly_trends')
print(f'gold_monthly_trends: {gold_monthly.count()} rows')
gold_monthly.show()

## Gold Table 3: Customer Segment Analysis

In [ ]:
gold_segments = fact_loans.groupBy('CustomerSegment','CreditScoreTier','AgeGroup') \
    .agg(
        F.count('LoanID').alias('NumLoans'),
        F.countDistinct('CustomerID').alias('UniqueCustomers'),
        F.round(F.avg('LoanAmount'), 0).alias('AvgLoanAmount'),
        F.round(F.avg('InterestRate'), 2).alias('AvgInterestRate'),
        F.round(F.sum('IsDefault') * 100.0 / F.count('LoanID'), 1).alias('DefaultRate_Pct'),
        F.round(F.sum('LoanAmount'), 0).alias('TotalLoanVolume')
    )

gold_segments.write.format('delta').mode('overwrite').saveAsTable('gold_customer_segments')
print(f'gold_customer_segments: {gold_segments.count()} rows')
gold_segments.orderBy('TotalLoanVolume', ascending=False).show()

## Gold Table 4: Product Performance

In [ ]:
gold_products = fact_loans.groupBy('ProductID','ProductName','ProductType') \
    .agg(
        F.count('LoanID').alias('NumLoans'),
        F.round(F.sum('LoanAmount'), 0).alias('TotalLoanVolume'),
        F.round(F.avg('LoanAmount'), 0).alias('AvgLoanAmount'),
        F.round(F.avg('InterestRate'), 2).alias('AvgInterestRate'),
        F.round(F.sum('OutstandingBalance'), 0).alias('TotalOutstanding'),
        F.sum('IsDefault').alias('Defaults'),
        F.round(F.sum('IsDefault') * 100.0 / F.count('LoanID'), 1).alias('DefaultRate_Pct')
    )

gold_products.write.format('delta').mode('overwrite').saveAsTable('gold_product_performance')
print(f'gold_product_performance: {gold_products.count()} rows')
gold_products.orderBy('TotalLoanVolume', ascending=False).show()

## Executive Summary Query — Cross all Gold Tables

In [ ]:
%%sql
-- Top 5 branches by loan volume with default rate
SELECT
    BranchName,
    Region,
    TotalLoans,
    CONCAT('$', FORMAT_NUMBER(TotalLoanVolume, 0))  AS LoanVolume,
    CONCAT(DefaultRate_Pct, '%')                    AS DefaultRate,
    CONCAT('$', FORMAT_NUMBER(TotalOutstanding, 0)) AS Outstanding
FROM gold_branch_performance
ORDER BY TotalLoanVolume DESC
LIMIT 5

In [ ]:
%%sql
-- All Gold tables created
SELECT 'gold_branch_performance' AS GoldTable, COUNT(*) AS Rows FROM gold_branch_performance UNION ALL
SELECT 'gold_monthly_trends',                  COUNT(*)         FROM gold_monthly_trends      UNION ALL
SELECT 'gold_customer_segments',               COUNT(*)         FROM gold_customer_segments   UNION ALL
SELECT 'gold_product_performance',             COUNT(*)         FROM gold_product_performance